<a href="https://colab.research.google.com/github/Magar-Bhuwan/ML-Internship-Assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Signal checks

**Signal 1 — Staleness (`days_since_last_update`)**

**Verdict: CONFIRMED**

The observed data shows a substantial drop in median impressions and clicks among the oldest content. The relationship is not perfectly monotonic across every bucket, but the 105–180 and 181+ day groups have much lower observed activity. This supports using staleness as a directional review signal and is aligned with the refresh-flag logic.

**Signal 2 — Visibility (`impressions_90d`)**

**Verdict: CONFIRMED**

Higher impression buckets show substantially higher observed clicks and CTR. This supports using 90-day impressions as a visibility and opportunity signal for prioritization.

### Baseline rule

Prioritize content for human review when it has not been updated for at least 104 days and has at least 3,616 impressions during the last 90 days. Among qualifying content, higher 90-day impressions receive higher priority.

The rule is decision-support, not a claim that stale content causes poor performance.

### Reason codes

* `stale_and_visible` — content is at least 104 days since its last update and has at least 3,616 impressions in the last 90 days.
* `stale` — content is at least 104 days since its last update but has lower visibility.
* `visible` — content has at least 3,616 impressions but is not stale.
* `no_strong_signal` — neither condition is met.

### Actions

* `review` — prioritize for human review.
* `monitor` — do not prioritize immediately, but retain in the ranked queue.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Build the ranked queue using a transparent rule. The score prioritizes content that is at least 104 days since its last update and has at least 3,616 impressions in the last 90 days. Higher impressions receive higher priority among qualifying content. The ranked queue is written to work/outputs/baseline_action_score.csv.

In [77]:
import pandas as pd

df = pd.read_csv("ML-Internship-Assignments/data/raw/content_refresh_anonymized.csv")

print(df.shape)

(30000, 44)


In [78]:
# Section 2 — Build the ranked queue

from pathlib import Path

# Thresholds based on the observed dataset
STALE_DAYS = 104
VISIBLE_IMPRESSIONS = 3616

# 1. Create the two rule signals
df["stale"] = (
    df["days_since_last_update"] >= STALE_DAYS
).astype(int)

df["visible"] = (
    df["impressions_90d"] >= VISIBLE_IMPRESSIONS
).astype(int)

# 2. Transparent baseline score
# Only stale + visible content receives a positive score.
# Higher impressions = higher priority.
df["score"] = (
    df["stale"]
    * df["visible"]
    * df["impressions_90d"]
)

# 3. Assign reason codes
df["reason_code"] = "no_strong_signal"

df.loc[
    (df["stale"] == 1) & (df["visible"] == 1),
    "reason_code"
] = "stale_and_visible"

df.loc[
    (df["stale"] == 1) & (df["visible"] == 0),
    "reason_code"
] = "stale"

df.loc[
    (df["stale"] == 0) & (df["visible"] == 1),
    "reason_code"
] = "visible"

# 4. Assign action labels
df["action"] = "monitor"

df.loc[
    df["reason_code"] == "stale_and_visible",
    "action"
] = "review"

# 5. Rank all content items
ranked = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).copy()

ranked["rank"] = range(1, len(ranked) + 1)

# 6. Select the columns for the ranked queue
output_columns = [
    "rank",
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "score",
    "reason_code",
    "action"
]

baseline_queue = ranked[output_columns].copy()

# 7. Write the required CSV
output_path = Path(
    "ML-Internship-Assignments/work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

baseline_queue.to_csv(
    output_path,
    index=False
)

# 8. Verify
print(f"Output saved to: {output_path}")
print(f"Rows written: {len(baseline_queue):,}")

print("\nTop 10:")
display(baseline_queue.head(10))

Output saved to: ML-Internship-Assignments/work/outputs/baseline_action_score.csv
Rows written: 30,000

Top 10:


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
6653,1,content_5fe46e04994d,104,517715,517715,stale_and_visible,review
29400,2,content_2dba2b1f9536,104,443434,443434,stale_and_visible,review
13537,3,content_2c2606c5d176,104,347399,347399,stale_and_visible,review
26531,4,content_cb112fce36be,104,309910,309910,stale_and_visible,review
21565,5,content_9532f197bbc8,104,309192,309192,stale_and_visible,review
3394,6,content_36ff89c8214e,104,295097,295097,stale_and_visible,review
26798,7,content_b28d1efd668f,104,286608,286608,stale_and_visible,review
23767,8,content_813e88069237,104,233561,233561,stale_and_visible,review
26255,9,content_c21024970297,104,211366,211366,stale_and_visible,review
7445,10,content_c8e9d6ab9013,104,208678,208678,stale_and_visible,review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 items are all assigned the `review` action with the `stale_and_visible` reason code. Each item has reached the 104-day staleness threshold and has substantial 90-day impressions. Because all top-20 items are exactly at the 104-day boundary, the ranking among them is primarily determined by impressions.

The review is intentionally skeptical. A high score means that an item matches the baseline rule strongly; it does not prove that the content needs to be refreshed. A manual review could show that the content is still accurate, intentionally evergreen, or not a useful refresh opportunity.


In [79]:
# Section 3 — Top-20 Review

top20_review = baseline_queue.head(20).copy()


def make_confidence_note(row):
    impressions = row["impressions_90d"]
    days = row["days_since_last_update"]

    if impressions >= 300000:
        level = "High"
    elif impressions >= 200000:
        level = "Moderate-high"
    else:
        level = "Moderate"

    return (
        f"{level} confidence in rule fit: the item is {days} days since "
        f"update and has {impressions:,} 90-day impressions."
    )


def make_wrong_note(row):
    impressions = row["impressions_90d"]

    if impressions >= 300000:
        return (
            "Could be wrong if the content is intentionally evergreen, "
            "still accurate, or high impressions do not indicate a useful "
            "refresh opportunity."
        )

    elif impressions >= 200000:
        return (
            "Could be wrong if the content remains accurate or a refresh "
            "would not improve its performance."
        )

    else:
        return (
            "Could be wrong if the content remains accurate and its "
            "visibility does not translate into a meaningful refresh "
            "opportunity."
        )


top20_review["confidence_note"] = top20_review.apply(
    make_confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    make_wrong_note,
    axis=1
)


# Final Top-20 review table
display(
    top20_review[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
6653,1,content_5fe46e04994d,review,stale_and_visible,High confidence in rule fit: the item is 104 d...,Could be wrong if the content is intentionally...
29400,2,content_2dba2b1f9536,review,stale_and_visible,High confidence in rule fit: the item is 104 d...,Could be wrong if the content is intentionally...
13537,3,content_2c2606c5d176,review,stale_and_visible,High confidence in rule fit: the item is 104 d...,Could be wrong if the content is intentionally...
26531,4,content_cb112fce36be,review,stale_and_visible,High confidence in rule fit: the item is 104 d...,Could be wrong if the content is intentionally...
21565,5,content_9532f197bbc8,review,stale_and_visible,High confidence in rule fit: the item is 104 d...,Could be wrong if the content is intentionally...
3394,6,content_36ff89c8214e,review,stale_and_visible,Moderate-high confidence in rule fit: the item...,Could be wrong if the content remains accurate...
26798,7,content_b28d1efd668f,review,stale_and_visible,Moderate-high confidence in rule fit: the item...,Could be wrong if the content remains accurate...
23767,8,content_813e88069237,review,stale_and_visible,Moderate-high confidence in rule fit: the item...,Could be wrong if the content remains accurate...
26255,9,content_c21024970297,review,stale_and_visible,Moderate-high confidence in rule fit: the item...,Could be wrong if the content remains accurate...
7445,10,content_c8e9d6ab9013,review,stale_and_visible,Moderate-high confidence in rule fit: the item...,Could be wrong if the content remains accurate...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline is intentionally simple, so some ranked items may be weak picks. A high score only means that an item is stale and has substantial observed visibility; it does not prove that the content needs a refresh.

The main weakness is that the rule uses only staleness and impressions. It does not know whether the content is still accurate, whether the topic is evergreen, or whether a refresh would actually improve performance.

The leakage check confirms that the baseline does not use `trend_pct`, `trend_direction`, or `is_declining_label`. It also does not use future-window information. The rule uses only `days_since_last_update` and `impressions_90d`, which are available in the current dataset snapshot.


In [80]:
# Identify weaker picks within the Top-20
weak_picks = top20_review.tail(3).copy()

weak_picks["weakness_reason"] = (
    "Lower ranked than the other Top-20 items because it has "
    "less observed 90-day visibility. The rule cannot determine "
    "whether a refresh would actually improve performance."
)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "score",
            "reason_code",
            "action",
            "weakness_reason"
        ]
    ]
)

,rank,content_id,impressions_90d,score,reason_code,action,weakness_reason
2267,18,content_f02b48f88241,181514,181514,stale_and_visible,review,Lower ranked than the other Top-20 items becau...
13370,19,content_05e9b4cd9ccf,179002,179002,stale_and_visible,review,Lower ranked than the other Top-20 items becau...
708,20,content_bb5bd5f771dc,176296,176296,stale_and_visible,review,Lower ranked than the other Top-20 items becau...


In [81]:
# Section 4 — Leakage Check

forbidden_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

baseline_features = [
    "days_since_last_update",
    "impressions_90d"
]

print("Baseline features used:")
for col in baseline_features:
    print(f"  ✓ {col}")

print("\nForbidden / label-derived columns:")
for col in forbidden_columns:
    if col in baseline_features:
        print(f"  ✗ LEAKAGE: {col}")
    else:
        print(f"  ✓ Not used: {col}")

Baseline features used:
  ✓ days_since_last_update
  ✓ impressions_90d

Forbidden / label-derived columns:
  ✓ Not used: trend_pct
  ✓ Not used: trend_direction
  ✓ Not used: is_declining_label


In [82]:
score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

print("Columns used to calculate the baseline score:")
print(score_inputs)

print("\nScore formula:")
print("score = stale × visible × impressions_90d")

Columns used to calculate the baseline score:
['days_since_last_update', 'impressions_90d']

Score formula:
score = stale × visible × impressions_90d


In [83]:
future_window_columns = [
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "future_trend",
    "future_label"
]

used_future_columns = [
    col for col in future_window_columns
    if col in baseline_features
]

print("Future-window columns used:", used_future_columns)

if len(used_future_columns) == 0:
    print("✓ No future-window columns used by the baseline.")
else:
    print("✗ Potential future-window leakage detected.")

Future-window columns used: []
✓ No future-window columns used by the baseline.


In [84]:
# Product / outcome-related columns that should not drive the baseline
flag_like_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

used_flag_columns = [
    col for col in flag_like_columns
    if col in baseline_features
]

print("Flag/label columns used by baseline:", used_flag_columns)

if len(used_flag_columns) == 0:
    print("✓ No product flags or label-derived columns were used.")
else:
    print("✗ Potential product-flag leakage detected.")

Flag/label columns used by baseline: []
✓ No product flags or label-derived columns were used.


### Section 4 conclusion

The weakest Top-20 picks are the lower-ranked items because they have less observed visibility than the highest-ranked items, although they still satisfy both baseline thresholds. Their inclusion is therefore a reasonable review candidate but not a guaranteed refresh opportunity.

The leakage checks passed. The baseline score uses only `days_since_last_update` and `impressions_90d`. It does not use `trend_pct`, `trend_direction`, `is_declining_label`, product flags, or future-window information. The baseline is therefore based on observable snapshot information rather than future outcomes or label-derived inputs.


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.